<a href="https://colab.research.google.com/github/AHichem/bridge-deterioration-nbi/blob/main/notebooks/02_sql_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 — Modèle de données SQL

Objectif : à partir de la table `inspections` nettoyée, construire un **panel pont × année** avec les événements de dégradation, plus les tables de dimensions.

- **Partie 0** (fournie) : setup + chargement/nettoyage → table `inspections`
- **Étapes 1 à 7** : c'est toi qui écris le SQL

## 0.1 — Setup

In [5]:
!pip install -q jupysql duckdb-engine

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/nbi'

import duckdb
con = duckdb.connect(f'{DATA_DIR}/nbi.duckdb')

%load_ext sql
%sql con
%config SqlMagic.displaylimit = 20

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


CatalogException: Catalog Error: Failure while replaying WAL file "/content/drive/MyDrive/nbi/nbi.duckdb.wal": Table with name dim_material does not exist!

## 0.2 — Chargement + nettoyage → `inspections`

Deux temps :
1. **Lecture (Python/pandas)** → `inspections_raw` : les 19 colonnes utiles, en texte brut, une année à la fois. On passe par pandas car le lecteur CSV de DuckDB n'arrive pas à lire certains fichiers NBI (fins de ligne mélangées `\n` / `\r\n`, apostrophes dans les textes) ; pandas est plus tolérant.
2. **Nettoyage (SQL)** → `inspections` : typage, conversion lat/lon, notes `'N'` → NULL, `min_rating`, `age`.

| Colonne | Contenu |
|---|---|
| `bridge_id` | `state_code` + `-` + `structure_number` |
| `year` | année du fichier NBI (2016–2025 ; le fichier 2015 téléchargé n'est pas exploitable) |
| `state_code`, `county_code`, `lat`, `lon` | localisation (lat/lon en degrés décimaux, lon négative) |
| `owner_code`, `material_code`, `structure_type_code` | codes bruts (items 22, 43A, 43B) |
| `year_built`, `year_reconstructed` | entiers, NULL si 0/absent |
| `adt`, `pct_truck`, `length_m`, `deck_width_m` | trafic et géométrie |
| `deck`, `super`, `sub`, `culvert` | notes 0–9, `'N'` → NULL |
| `is_culvert` | `culvert IS NOT NULL` |
| `min_rating` | `culvert` si ponceau, sinon `LEAST(deck, super, sub)` |
| `age` | `year - year_built` |

Choix de nettoyage :
- on garde seulement `RECORD_TYPE_005A = '1'` (les lignes de type 2 décrivent la route *sous* le pont et dupliqueraient la structure) ;
- les doublons éventuels ne sont **pas** supprimés ici : tu les chercheras à l'étape 6.

⏱️ Lecture de ~2,5 Go depuis le Drive : compte plusieurs minutes.

In [ ]:
# Lecture des fichiers bruts avec pandas, un fichier (= une année) à la fois.
# pandas tolère les fins de ligne mélangées (\n / \r\n) qui font planter le lecteur CSV de DuckDB.
import glob, os, re
import pandas as pd

COLS = [
    'STATE_CODE_001', 'STRUCTURE_NUMBER_008', 'RECORD_TYPE_005A', 'COUNTY_CODE_003',
    'LAT_016', 'LONG_017', 'OWNER_022', 'STRUCTURE_KIND_043A', 'STRUCTURE_TYPE_043B',
    'YEAR_BUILT_027', 'YEAR_RECONSTRUCTED_106', 'ADT_029', 'PERCENT_ADT_TRUCK_109',
    'STRUCTURE_LEN_MT_049', 'DECK_WIDTH_MT_052',
    'DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062',
]

con.execute('DROP TABLE IF EXISTS inspections_raw')
for f in sorted(glob.glob(f'{DATA_DIR}/raw/nbi_20*.txt')):
    year = int(re.search(r'nbi_(20\d{2})', f).group(1))
    if year < 2016:          # nbi_2015.txt n'est pas un vrai fichier NBI
        continue
    df = pd.read_csv(
        f, sep=',', quotechar="'", encoding='latin-1',
        dtype=object, keep_default_na=False, na_values=[''],   # tout en texte, vide -> NULL
        usecols=lambda c: c.strip() in COLS,
        on_bad_lines='warn',                                # signale les lignes illisibles au lieu de planter
    )
    df.columns = df.columns.str.strip()
    df = df.reindex(columns=COLS)                           # colonne absente une année -> NULL
    df.insert(0, 'year', year)
    if con.sql("SELECT COUNT(*) FROM information_schema.tables WHERE table_name = 'inspections_raw'").fetchone()[0] == 0:
        con.execute('CREATE TABLE inspections_raw AS SELECT * FROM df')
    else:
        con.execute('INSERT INTO inspections_raw SELECT * FROM df')
    print(f'{year} : {len(df):>8,} lignes')
    del df

print(con.sql('SELECT COUNT(*) AS total FROM inspections_raw').fetchone()[0], 'lignes au total')

In [ ]:
%%sql
-- Nettoyage + typage : inspections_raw (texte brut) -> inspections (table propre)
CREATE OR REPLACE TABLE inspections AS
WITH typed AS (
    SELECT
        lpad(trim(STATE_CODE_001), 2, '0') || '-' || trim(STRUCTURE_NUMBER_008) AS bridge_id,
        CAST(year AS INTEGER) AS year,
        lpad(trim(STATE_CODE_001), 2, '0')  AS state_code,
        lpad(trim(COUNTY_CODE_003), 3, '0') AS county_code,

        -- LAT_016 = DDMMSSss, LONG_017 = DDDMMSSss (ouest, donc négatif)
        lpad(trim(LAT_016), 8, '0')  AS lat_txt,
        lpad(trim(LONG_017), 9, '0') AS lon_txt,

        lpad(trim(OWNER_022), 2, '0')            AS owner_code,
        trim(STRUCTURE_KIND_043A)                AS material_code,
        lpad(trim(STRUCTURE_TYPE_043B), 2, '0')  AS structure_type_code,

        NULLIF(TRY_CAST(trim(YEAR_BUILT_027)         AS INTEGER), 0) AS year_built,
        NULLIF(TRY_CAST(trim(YEAR_RECONSTRUCTED_106) AS INTEGER), 0) AS year_reconstructed,

        TRY_CAST(trim(ADT_029)               AS INTEGER) AS adt,
        TRY_CAST(trim(PERCENT_ADT_TRUCK_109) AS INTEGER) AS pct_truck,
        TRY_CAST(trim(STRUCTURE_LEN_MT_049)  AS DOUBLE)  AS length_m,
        TRY_CAST(trim(DECK_WIDTH_MT_052)     AS DOUBLE)  AS deck_width_m,

        -- notes 0-9 ; 'N' (non applicable) ou vide -> NULL
        TRY_CAST(trim(DECK_COND_058)           AS TINYINT) AS deck,
        TRY_CAST(trim(SUPERSTRUCTURE_COND_059) AS TINYINT) AS super,
        TRY_CAST(trim(SUBSTRUCTURE_COND_060)   AS TINYINT) AS sub,
        TRY_CAST(trim(CULVERT_COND_062)        AS TINYINT) AS culvert
    FROM inspections_raw
    WHERE trim(RECORD_TYPE_005A) = '1'   -- 1 = la structure elle-même (2 = route qui passe dessous, doublon)
)
SELECT
    bridge_id, year, state_code, county_code,
    CASE WHEN TRY_CAST(lat_txt AS BIGINT) > 0 THEN
        CAST(substr(lat_txt, 1, 2) AS INTEGER)
      + CAST(substr(lat_txt, 3, 2) AS INTEGER) / 60.0
      + CAST(substr(lat_txt, 5, 4) AS INTEGER) / 100.0 / 3600.0
    END AS lat,
    CASE WHEN TRY_CAST(lon_txt AS BIGINT) > 0 THEN
      -( CAST(substr(lon_txt, 1, 3) AS INTEGER)
       + CAST(substr(lon_txt, 4, 2) AS INTEGER) / 60.0
       + CAST(substr(lon_txt, 6, 4) AS INTEGER) / 100.0 / 3600.0 )
    END AS lon,
    owner_code, material_code, structure_type_code,
    year_built, year_reconstructed,
    adt, pct_truck, length_m, deck_width_m,
    deck, super, sub, culvert,
    culvert IS NOT NULL AS is_culvert,
    CASE WHEN culvert IS NOT NULL THEN culvert
         ELSE LEAST(deck, super, sub)          -- LEAST ignore les NULL dans DuckDB
    END AS min_rating,
    year - year_built AS age
FROM typed;

In [ ]:
%%sql
-- Vérification rapide du chargement (pas l'étape 1 : juste s'assurer que la table existe)
SELECT COUNT(*) AS n_rows, MIN(year) AS first_year, MAX(year) AS last_year
FROM inspections;

In [ ]:
%%sql
SELECT * FROM inspections LIMIT 5;

In [ ]:
%%sql
-- SELECT * FROM (describe inspections) ;
summarize inspections

---
## Étape 1 — Prise en main de la table (15 min)

**Objectif :** répondre à 4 questions, une requête chacune.
1. Combien de lignes (= ponts inspectés) par année ?
2. Combien de ponts **distincts** au total sur toute la période ?
3. Distribution de `min_rating` par année (combien de ponts à 0, 1, …, 9 chaque année).
4. Part de ponceaux (`is_culvert`) — globale ou par année.

**Concepts :** `GROUP BY`, `COUNT(DISTINCT …)`, agrégats conditionnels `COUNT(*) FILTER (WHERE …)`.

**Indices (sans solution) :**
- Q1 et Q2 ne donnent pas le même ordre de grandeur que la somme des lignes : pourquoi ?
- Q3 : deux façons possibles — format « long » (`GROUP BY` sur deux colonnes) ou format « large » (une colonne par note avec `FILTER`). Essaie au moins le format large pour pratiquer `FILTER`.
- Q4 : un booléen se compte avec `FILTER (WHERE is_culvert)`, et une part = compte filtré / compte total. Attention à la division entière.
- Pense aux `NULL` dans `min_rating` : où vont-ils dans tes résultats ?

In [ ]:
%%sql
-- Number of bridges inspected by year
SELECT COUNT (DISTINCT bridge_id) AS nb_bridges_pery, year FROM inspections
GROUP BY year,
ORDER BY year;

In [ ]:
%%sql
-- Nb of bridges
SELECT COUNT (DISTINCT bridge_id) AS distinct_bridges FROM inspections;


In [ ]:
%%sql
-- Nb of bridges with a min rate per year
SELECT year,
COUNT (min_rating) FILTER  (WHERE min_rating = 0) AS minrat_0,
COUNT (min_rating) FILTER  (WHERE min_rating = 1) AS minrat_1,
COUNT (min_rating) FILTER  (WHERE min_rating = 2) AS minrat_2,
COUNT (min_rating) FILTER  (WHERE min_rating = 3) AS minrat_3,
COUNT (min_rating) FILTER  (WHERE min_rating = 4) AS minrat_4,
COUNT (min_rating) FILTER  (WHERE min_rating = 5) AS minrat_5,
COUNT (min_rating) FILTER  (WHERE min_rating = 6) AS minrat_6,
COUNT (min_rating) FILTER  (WHERE min_rating = 7) AS minrat_7,
COUNT (min_rating) FILTER  (WHERE min_rating = 8) AS minrat_8,
COUNT (min_rating) FILTER  (WHERE min_rating = 9) AS minrat_9
FROM inspections
GROUP BY year,
ORDER BY year;



In [ ]:
%%sql
-- Q4 — part de ponceaux
SELECT COUNT (DISTINCT bridge_id) AS per_culvert_pery, year FROM inspections
GROUP BY year,
ORDER BY year;


In [ ]:
%%sql
-- Percentage of culvert per year
SELECT ROUND( COUNT (is_culvert) FILTER (WHERE is_culvert==1) / COUNT (*) *100,2) AS per_culvert_pery, year FROM inspections
GROUP BY year,
ORDER BY year;


---
## LOOKUP TABLES



In [ ]:
%%sql
CREATE OR REPLACE TABLE dim_material(
    material_code	VARCHAR,
    material_label VARCHAR
);

INSERT INTO dim_material
  (material_code, material_label)
VALUES
    ('1', 'Concrete'),
    ('2', 'Concrete continuous'),
    ('3', 'Steel'),
    ('4', 'Steel continuous'),
    ('5', 'Prestressed concrete'),
    ('6', 'Prestressed concrete continuous'),
    ('7', 'Wood or Timber'),
    ('8', 'Masonry'),
    ('9', 'Aluminum, Wrought Iron or Cast Iron'),
    ('0', 'Other'),
    ('02', 'Concrete continuous'),   -- fix mistaken in table
    ('A',  'Invalid code');          -- fix Invalid code



In [ ]:
%%sql
-- orphan keys check
SELECT i.material_code, COUNT(*) AS n_rows
FROM inspections AS i
LEFT JOIN  dim_material AS d
ON i.material_code = d.material_code
WHERE d.material_code  IS NULL
GROUP BY i.material_code
ORDER BY n_rows;


In [ ]:
%%sql
CREATE OR REPLACE TABLE dim_owner(
    owner_code	VARCHAR,
    owner_label VARCHAR
);

INSERT INTO dim_owner
  (owner_code, owner_label)
VALUES
    ('01', 'State Highway Agency'),
    ('02', 'County Highway Agency'),
    ('03', 'Town or Township Highway Agency'),
    ('04', 'City or Municipal Highway Agency'),
    ('11', 'State Park, Forest, or Reservation Agency'),
    ('12', 'Local Park, Forest, or Reservation Agency'),
    ('21', 'Other State Agencies'),
    ('25', 'Other Local Agencies'),
    ('26', 'Private (other than railroad)'),
    ('27', 'Railroad'),
    ('31', 'State Toll Authority'),
    ('32', 'Local Toll Authority'),
    ('60', 'Other Federal Agencies'),
    ('61', 'Indian Tribal Government'),
    ('62', 'Bureau of Indian Affairs'),
    ('63', 'Bureau of Fish and Wildlife'),
    ('64', 'U.S. Forest Service'),
    ('66', 'National Park Service'),
    ('67', 'Tennessee Valley Authority'),
    ('68', 'Bureau of Land Management'),
    ('69', 'Bureau of Reclamation'),
    ('70', 'Corps of Engineers (Civil)'),
    ('71', 'Corps of Engineers (Military)'),
    ('72', 'Air Force'),
    ('73', 'Navy/Marines'),
    ('74', 'Army'),
    ('75', 'NASA'),
    ('76', 'Metropolitan Washington Airports Service'),
    ('80', 'Unknown'),
    -- fix Invalid code
    ('00', 'Invalid code'),
    ('06', 'Invalid code'),
    ('19', 'Invalid code'),
    ('40', 'Invalid code'),
    ('52', 'Invalid code'),
    ('53', 'Invalid code'),
    ('54', 'Invalid code'),
    ('55', 'Invalid code'),
    ('56', 'Invalid code'),
    ('57', 'Invalid code'),
    ('58', 'Invalid code'),
    ('59', 'Invalid code'),
    ('77', 'Invalid code'),
    ('78', 'Invalid code'),
    ('79', 'Invalid code');

In [ ]:
%%sql
--orphan keys check
SELECT i.owner_code, COUNT(*) AS n_rows
FROM inspections AS i
LEFT JOIN dim_owner AS d
    ON i.owner_code = d.owner_code
WHERE d.owner_code IS NULL
GROUP BY i.owner_code
ORDER BY n_rows DESC;

In [ ]:
%%sql
CREATE OR REPLACE TABLE dim_state (
    state_code VARCHAR,
    state_abbr VARCHAR,
    state_name VARCHAR
);

INSERT INTO dim_state
  (state_code, state_abbr, state_name)
VALUES
    ('01', 'AL', 'Alabama'),
    ('02', 'AK', 'Alaska'),
    ('04', 'AZ', 'Arizona'),
    ('05', 'AR', 'Arkansas'),
    ('06', 'CA', 'California'),
    ('08', 'CO', 'Colorado'),
    ('09', 'CT', 'Connecticut'),
    ('10', 'DE', 'Delaware'),
    ('11', 'DC', 'District of Columbia'),
    ('12', 'FL', 'Florida'),
    ('13', 'GA', 'Georgia'),
    ('15', 'HI', 'Hawaii'),
    ('16', 'ID', 'Idaho'),
    ('17', 'IL', 'Illinois'),
    ('18', 'IN', 'Indiana'),
    ('19', 'IA', 'Iowa'),
    ('20', 'KS', 'Kansas'),
    ('21', 'KY', 'Kentucky'),
    ('22', 'LA', 'Louisiana'),
    ('23', 'ME', 'Maine'),
    ('24', 'MD', 'Maryland'),
    ('25', 'MA', 'Massachusetts'),
    ('26', 'MI', 'Michigan'),
    ('27', 'MN', 'Minnesota'),
    ('28', 'MS', 'Mississippi'),
    ('29', 'MO', 'Missouri'),
    ('30', 'MT', 'Montana'),
    ('31', 'NE', 'Nebraska'),
    ('32', 'NV', 'Nevada'),
    ('33', 'NH', 'New Hampshire'),
    ('34', 'NJ', 'New Jersey'),
    ('35', 'NM', 'New Mexico'),
    ('36', 'NY', 'New York'),
    ('37', 'NC', 'North Carolina'),
    ('38', 'ND', 'North Dakota'),
    ('39', 'OH', 'Ohio'),
    ('40', 'OK', 'Oklahoma'),
    ('41', 'OR', 'Oregon'),
    ('42', 'PA', 'Pennsylvania'),
    ('44', 'RI', 'Rhode Island'),
    ('45', 'SC', 'South Carolina'),
    ('46', 'SD', 'South Dakota'),
    ('47', 'TN', 'Tennessee'),
    ('48', 'TX', 'Texas'),
    ('49', 'UT', 'Utah'),
    ('50', 'VT', 'Vermont'),
    ('51', 'VA', 'Virginia'),
    ('53', 'WA', 'Washington'),
    ('54', 'WV', 'West Virginia'),
    ('55', 'WI', 'Wisconsin'),
    ('56', 'WY', 'Wyoming'),
    ('66', 'GU', 'Guam'),
    ('72', 'PR', 'Puerto Rico'),
    ('78', 'VI', 'U.S. Virgin Islands');

In [ ]:
%%sql
-- orphan keys check
SELECT i.state_code, COUNT(*) AS n_rows
FROM inspections AS i
LEFT JOIN dim_state AS d
    ON i.state_code = d.state_code
WHERE d.state_code IS NULL
GROUP BY i.state_code
ORDER BY n_rows DESC;

In [ ]:
%%sql
CREATE OR REPLACE TABLE dim_bridge AS
SELECT
    bridge_id,
    state_code,
    county_code,
    lat,
    lon,
    owner_code,
    material_code,
    structure_type_code,
    year_built,
    year_reconstructed,
    length_m,
    deck_width_m,
    is_culvert,
    MIN(year) OVER (PARTITION BY bridge_id)  AS first_year,
    year                                 AS last_year
FROM inspections
QUALIFY ROW_NUMBER() OVER (
            PARTITION BY bridge_id
            ORDER BY year DESC, year_reconstructed DESC NULLS LAST
        ) = 1;



In [ ]:
%%sql
select count (*)
from dim_bridge
groupby bridge_id

In [ ]:
%%sql
SELECT bridge_id, COUNT(*) AS n
FROM dim_bridge
GROUP BY bridge_id
HAVING COUNT(*) > 1;

